In [ ]:
import logging
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

logger = logging.getLogger("CrossEvaluation")
if not logger.handlers:
    _handler = logging.StreamHandler()
    _handler.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(_handler)
logger.setLevel(logging.INFO)
logger.propagate = False


def safe_parse_sensor_set(config_str: object) -> frozenset[str]:
    if pd.isna(config_str):
        return frozenset()
    s = str(config_str).strip()
    if s.lower().startswith("frozenset(") and s.endswith(")"):
        s = s[len("frozenset(") : -1].strip()
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")) or (s.startswith("(") and s.endswith(")")):
        s = s[1:-1].strip()
    if not s:
        return frozenset()
    if "," in s:
        parts = [p.strip() for p in s.split(",")]
    elif ";" in s:
        parts = [p.strip() for p in s.split(";")]
    else:
        parts = [p.strip() for p in s.split()]
    elements: list[str] = []
    for part in parts:
        cleaned = part.strip().strip("'").strip('"')
        if cleaned:
            elements.append(str(cleaned))
    return frozenset(elements)


def calculate_binary_score(sensor_set: frozenset[str], binary_matrix_data: np.ndarray, node_map: dict[str, int]) -> float:
    if not sensor_set:
        return 0.0 if binary_matrix_data is not None and binary_matrix_data.shape[0] > 0 else np.nan
    if binary_matrix_data is None or node_map is None or binary_matrix_data.shape[0] == 0:
        return np.nan
    valid_idx = [node_map[str(sid)] for sid in sensor_set if str(sid) in node_map]
    valid_idx = [i for i in valid_idx if i < binary_matrix_data.shape[1]]
    if not valid_idx:
        return 0.0 if binary_matrix_data.shape[0] > 0 else np.nan
    return float(np.mean(np.any(binary_matrix_data[:, valid_idx], axis=1)))


def calculate_sigmoid_score(sensor_set: frozenset[str], sigmoid_matrix_data: np.ndarray, node_map: dict[str, int]) -> float:
    if not sensor_set:
        return 0.0 if sigmoid_matrix_data is not None and sigmoid_matrix_data.shape[0] > 0 else np.nan
    if sigmoid_matrix_data is None or node_map is None or sigmoid_matrix_data.shape[0] == 0:
        return np.nan
    valid_idx = [node_map[str(sid)] for sid in sensor_set if str(sid) in node_map]
    valid_idx = [i for i in valid_idx if i < sigmoid_matrix_data.shape[1]]
    if not valid_idx:
        return 0.0 if sigmoid_matrix_data.shape[0] > 0 else np.nan
    return float(np.mean(np.max(sigmoid_matrix_data[:, valid_idx], axis=1)))


def discover_models(tbpo_dir: Path, cspo_dir: Path) -> list[dict[str, object]]:
    models: list[dict[str, object]] = []
    tbpo_pattern = re.compile(r"T(\d+\.\d+)_pareto_optimal_solutions_summary\.csv")

    if tbpo_dir.exists() and tbpo_dir.is_dir():
        for csv_path in tbpo_dir.glob("*_pareto_optimal_solutions_summary.csv"):
            m = tbpo_pattern.match(csv_path.name)
            if not m:
                continue
            thr = m.group(1)
            models.append(
                {
                    "name": f"DP_{thr}",
                    "type": "binary",
                    "threshold": float(thr),
                    "solutions_csv": csv_path,
                    "matrix_parquet": Path(f"T{thr}_Binary_Judgment_Matrix_Processed.parquet"),
                }
            )
        thet_csv = tbpo_dir / "Thet_pareto_optimal_solutions_summary.csv"
        if thet_csv.exists():
            models.append(
                {
                    "name": "DP_het",
                    "type": "binary",
                    "threshold": None,
                    "solutions_csv": thet_csv,
                    "matrix_parquet": Path("Thet_Binary_Judgment_Matrix_Processed.parquet"),
                }
            )

    def _sort_key(d: dict[str, object]) -> float:
        thr = d.get("threshold")
        return float("inf") if thr is None else float(thr)

    models.sort(key=_sort_key)

    cspo_solution = cspo_dir / "pareto_optimal_solutions_summary.csv"
    if cspo_solution.exists():
        models.append(
            {
                "name": "CSPO",
                "type": "sigmoid",
                "threshold": None,
                "solutions_csv": cspo_solution,
                "matrix_parquet": Path("Sigmoid_Detection_Matrix_Processed.parquet"),
            }
        )

    return models


class CrossModelEvaluator:
    def __init__(self, models: list[dict[str, object]], judgment_matrix_dir: Path, sigmoid_matrix_dir: Path, output_dir: Path):
        self.models = models
        self.judgment_matrix_dir = judgment_matrix_dir
        self.sigmoid_matrix_dir = sigmoid_matrix_dir
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.solutions_by_model: dict[str, pd.DataFrame] = {}
        self.native_perf_col_by_model: dict[str, str] = {}
        self.matrices_by_model: dict[str, dict[str, object]] = {}
        self.layouts: list[dict[str, object]] = []

    def load_solutions(self) -> None:
        for m in self.models:
            name = str(m["name"])
            model_type = str(m["type"])
            solution_path: Path = m["solutions_csv"]  # type: ignore[assignment]
            if not solution_path.exists():
                raise FileNotFoundError(f"Missing solutions: {solution_path}")

            df = pd.read_csv(solution_path)
            if "Num_Sensors" in df.columns and "Number_of_Sensors" not in df.columns:
                df.rename(columns={"Num_Sensors": "Number_of_Sensors"}, inplace=True)
            if "Sensor_Set_Str" in df.columns and "Selected_Sensor_IDs" not in df.columns:
                df.rename(columns={"Sensor_Set_Str": "Selected_Sensor_IDs"}, inplace=True)

            required = ["Number_of_Sensors", "Selected_Sensor_IDs"]
            missing = [c for c in required if c not in df.columns]
            if missing:
                raise KeyError(f"Missing columns {missing} for '{name}': {solution_path}")

            if model_type == "binary":
                perf_col = "DP"
            elif model_type == "sigmoid":
                perf_col = "CDS"
            else:
                raise ValueError(f"Unknown model type: {model_type}")
            if perf_col not in df.columns:
                raise KeyError(f"Missing '{perf_col}' column for '{name}': {solution_path}")

            self.solutions_by_model[name] = df
            self.native_perf_col_by_model[name] = perf_col

        if not self.solutions_by_model:
            raise RuntimeError("No solutions loaded.")

    def load_matrices(self) -> None:
        for m in self.models:
            name = str(m["name"])
            model_type = str(m["type"])
            matrix_filename: Path = m["matrix_parquet"]  # type: ignore[assignment]
            matrix_dir = self.sigmoid_matrix_dir if model_type == "sigmoid" else self.judgment_matrix_dir
            matrix_path = matrix_dir / matrix_filename
            if not matrix_path.exists():
                raise FileNotFoundError(f"Missing matrix for '{name}': {matrix_path}")

            df_matrix = pq.read_table(matrix_path).to_pandas()
            if df_matrix.empty or df_matrix.shape[1] < 2:
                raise ValueError(f"Matrix invalid for '{name}': {matrix_path}")

            node_names = df_matrix.columns[1:].astype(str).tolist()
            node_map = {n: i for i, n in enumerate(node_names)}
            data = df_matrix.iloc[:, 1:].values
            self.matrices_by_model[name] = {
                "type": model_type,
                "threshold": m.get("threshold"),
                "node_names": node_names,
                "node_map": node_map,
                "data": data,
                "path": matrix_path,
            }

        if not self.matrices_by_model:
            raise RuntimeError("No matrices loaded.")

        if len(self.matrices_by_model) > 1:
            first_key = next(iter(self.matrices_by_model))
            base_nodes = set(self.matrices_by_model[first_key]["node_names"])
            for k, info in self.matrices_by_model.items():
                if set(info["node_names"]) != base_nodes:
                    raise ValueError(f"Node names mismatch between '{first_key}' and '{k}'.")

    def collect_layouts(self) -> None:
        layouts: list[dict[str, object]] = []
        for model_name, df in self.solutions_by_model.items():
            perf_col = self.native_perf_col_by_model[model_name]
            for _, row in df.iterrows():
                k = int(row["Number_of_Sensors"])
                sensor_str = row["Selected_Sensor_IDs"]
                layout = safe_parse_sensor_set(sensor_str)
                if not layout and k > 0:
                    continue
                layouts.append(
                    {
                        "layout": layout,
                        "origin_model": model_name,
                        "k": k,
                        "origin_native_performance": row[perf_col],
                        "sensor_str": str(sensor_str),
                    }
                )
        if not layouts:
            raise RuntimeError("No layouts collected.")
        self.layouts = layouts

    def cross_evaluate(self) -> pd.DataFrame:
        rows: list[dict[str, object]] = []
        for layout_item in self.layouts:
            layout = layout_item["layout"]
            origin_model = layout_item["origin_model"]
            k = layout_item["k"]
            origin_perf = layout_item["origin_native_performance"]
            sensor_str = layout_item["sensor_str"]
            for eval_model, m in self.matrices_by_model.items():
                m_type = str(m["type"])
                data = m["data"]
                node_map = m["node_map"]
                if m_type == "binary":
                    score = calculate_binary_score(layout, data, node_map)
                else:
                    score = calculate_sigmoid_score(layout, data, node_map)
                rows.append(
                    {
                        "Original_Sensor_IDs_Str": sensor_str,
                        "Sensor_Count": k,
                        "Origin_Model_Name": origin_model,
                        "Origin_Native_Performance": origin_perf,
                        "Evaluation_Model_Name": eval_model,
                        "Evaluation_Matrix_Type": m_type,
                        "Evaluation_Matrix_Threshold": m.get("threshold"),
                        "Reevaluated_Performance": score,
                    }
                )

        eval_df = pd.DataFrame(rows)
        eval_df["Sensor_Count"] = pd.to_numeric(eval_df["Sensor_Count"], errors="coerce").astype("Int64")

        benchmark = (
            pd.DataFrame(self.layouts)
            .groupby(["origin_model", "k"])["origin_native_performance"]
            .mean()
            .reset_index()
            .rename(
                columns={
                    "origin_model": "Evaluation_Model_Name",
                    "k": "Sensor_Count",
                    "origin_native_performance": "Benchmark_Performance_of_Eval_Model",
                }
            )
        )
        eval_df = pd.merge(eval_df, benchmark, on=["Evaluation_Model_Name", "Sensor_Count"], how="left")
        denom = eval_df["Benchmark_Performance_of_Eval_Model"]
        eval_df["Ratio_vs_Eval_Model_Native_Avg"] = np.where(
            denom.notna() & (denom.abs() > 1e-9),
            eval_df["Reevaluated_Performance"] / denom,
            np.nan,
        )
        eval_df["Ratio_vs_Eval_Model_Native_Avg"] = eval_df["Ratio_vs_Eval_Model_Native_Avg"].replace([np.inf, -np.inf], np.nan)
        return eval_df

    def save_outputs(self, eval_df: pd.DataFrame) -> None:
        eval_df = eval_df.copy()
        eval_df["Reevaluated_Performance"] = pd.to_numeric(eval_df["Reevaluated_Performance"], errors="coerce")
        eval_df["Sensor_Count"] = pd.to_numeric(eval_df["Sensor_Count"], errors="coerce").astype("Int64")
        eval_df["Ratio_vs_Eval_Model_Native_Avg"] = pd.to_numeric(eval_df["Ratio_vs_Eval_Model_Native_Avg"], errors="coerce")

        perf_mean = (
            eval_df.groupby(["Origin_Model_Name", "Sensor_Count", "Evaluation_Model_Name"])["Reevaluated_Performance"]
            .mean()
            .reset_index(name="Mean_Reevaluated_Performance")
        )

        eval_order = [str(m["name"]) for m in self.models]
        origin_order = [str(m["name"]) for m in self.models]
        pivot = perf_mean.pivot_table(
            index=["Evaluation_Model_Name", "Sensor_Count"],
            columns="Origin_Model_Name",
            values="Mean_Reevaluated_Performance",
        )
        unique_k = sorted(perf_mean["Sensor_Count"].dropna().unique().astype(int))
        desired_index = pd.MultiIndex.from_product([eval_order, unique_k], names=["Evaluation_Model_Name", "Sensor_Count"])
        pivot = pivot.reindex(index=desired_index)
        pivot = pivot.reindex(columns=[c for c in origin_order if c in pivot.columns])
        pivot.dropna(how="all", axis=0, inplace=True)
        pivot.reset_index(inplace=True)
        pivot.to_csv(
            self.output_dir / "cross_evaluation.csv",
            index=False,
            float_format="%.5f",
            encoding="utf-8-sig",
            lineterminator="\n",
        )

        binary_eval_models = [str(m["name"]) for m in self.models if m["type"] == "binary"]
        rpi_by_eval = (
            eval_df[eval_df["Evaluation_Model_Name"].isin(binary_eval_models)]
            .groupby(["Origin_Model_Name", "Sensor_Count", "Evaluation_Model_Name"])["Ratio_vs_Eval_Model_Native_Avg"]
            .mean()
            .reset_index(name="Mean_RPI")
        )
        if rpi_by_eval.empty:
            raise RuntimeError("No data for robustness assessment (binary evaluation models).")

        grouped = rpi_by_eval.groupby(["Origin_Model_Name", "Sensor_Count"])["Mean_RPI"]
        robustness_df = grouped.agg(Overall_Mean_RPI="mean", Std_RPI="std").reset_index()
        robustness_df["CV_RPI"] = np.where(
            robustness_df["Overall_Mean_RPI"].notna() & (robustness_df["Overall_Mean_RPI"].abs() > 1e-9),
            robustness_df["Std_RPI"] / robustness_df["Overall_Mean_RPI"],
            np.nan,
        )
        robustness_df.loc[
            (robustness_df["Std_RPI"].abs() < 1e-9) & (robustness_df["Overall_Mean_RPI"].abs() < 1e-9), "CV_RPI"
        ] = 0.0
        robustness_df.loc[
            (robustness_df["Std_RPI"].abs() < 1e-9) & (robustness_df["Overall_Mean_RPI"].abs() >= 1e-9), "CV_RPI"
        ] = 0.0

        def format_method(name: str) -> str:
            if name == "CSPO":
                return "CSPO"
            if name == "DP_het":
                return "TBPO_het"
            if name.startswith("DP_"):
                return name.replace("DP_", "TBPO_", 1)
            return name

        out = pd.DataFrame(
            {
                "Method": robustness_df["Origin_Model_Name"].astype(str).map(format_method),
                "Number_of_Sensors": robustness_df["Sensor_Count"],
                "mu_RPI": robustness_df["Overall_Mean_RPI"],
                "CV_RPI": robustness_df["CV_RPI"],
            }
        )

        method_order = [format_method(str(m["name"])) for m in self.models]
        out["Method"] = pd.Categorical(out["Method"], categories=method_order, ordered=True)
        out.sort_values(["Method", "Number_of_Sensors"], inplace=True)
        out["Method"] = out["Method"].astype(str)

        out.to_csv(
            self.output_dir / "robustness_assessment.csv",
            index=False,
            float_format="%.5f",
            encoding="utf-8-sig",
            lineterminator="\n",
        )

    def run(self) -> None:
        self.load_solutions()
        self.load_matrices()
        self.collect_layouts()
        eval_df = self.cross_evaluate()
        if eval_df.empty:
            raise RuntimeError("No evaluation results.")
        self.save_outputs(eval_df)


base_dir = Path("output")
tbpo_dir = base_dir / "TBPO"
cspo_dir = base_dir / "CSPO"
judgment_matrix_dir = base_dir / "Judgment_Matrix"
sigmoid_matrix_dir = base_dir / "Sigmoid_Matrix"
analysis_output_dir = base_dir / "Cross_Evaluation"

models = discover_models(tbpo_dir=tbpo_dir, cspo_dir=cspo_dir)
if not models:
    raise RuntimeError("No models discovered. Check output/TBPO and output/CSPO.")

evaluator = CrossModelEvaluator(
    models=models,
    judgment_matrix_dir=judgment_matrix_dir,
    sigmoid_matrix_dir=sigmoid_matrix_dir,
    output_dir=analysis_output_dir,
)
evaluator.run()
logger.info(f"Wrote: {analysis_output_dir}")